#### **OPTIMALIDAD DE LA SOLUCION - ALGORITMOS GREEDY**

In [43]:
import pandas as pd 
import matplotlib.pyplot as plt
from importlib import reload

import Clases.asignacion as asignacion_module
reload(asignacion_module)
from Clases.asignacion import Asignacion

import Clases.caja as caja_module
reload(caja_module)
from Clases.caja import Caja

import Clases.producto as producto_module
reload(producto_module)
from Clases.producto import Producto

import Clases.solucion as solucion_module
reload(solucion_module)
from Clases.solucion import Solucion

catalogo_productos = pd.read_csv("Datos-finales/catalogo_productos.csv")
especificaciones_cajas = pd.read_csv("Datos-finales/especificaciones_cajas.csv")
operaciones_planta = pd.read_csv("Datos-finales/operaciones_planta.csv")
procurement_cajas = pd.read_csv("Datos-finales/procurement_cajas.csv")
factibilidad = pd.read_csv("Factibilidad/factibilidad_3mm.csv")

Empecemos guardando los productos y tipos de cajas en listas en el estado actual, para cargarlos luego a las soluciones. Calculamos también las cajas asignables a cada producto en la lista de productos.

In [44]:
def guardar_cajas_y_productos(grosor):
    caja_compras_merge = especificaciones_cajas.merge(procurement_cajas,
                                                  on="caja_tipo_id")
    cajas = [
        Caja(
            caja_id = row["caja_tipo_id"],
            dim_interior_ancho = row["caja_interior_ancho"],
            dim_interior_largo = row["caja_interior_largo"],
            dim_interior_alto = row["caja_interior_alto"]
        )
        for _, row in caja_compras_merge.iterrows()
    ]

    prod_op_merge = catalogo_productos.merge(operaciones_planta, on="codigo_producto")
    productos = [
        Producto(
            codigo_producto = row['codigo_producto'],
            cantidad_paquetes = row['cantidad_paquetes'],
            peso_paquete = row['peso_neto_paquete'],
            demanda_buenos_aires = row['volumen_producto_planta_buenos_aires'],
            demanda_curitiba = row['volumen_producto_planta_curitiba'],
            demanda_santiago = row['volumen_producto_planta_santiago'],
            demanda_monterrey = row['volumen_producto_planta_monterrey'],
            demanda_bakersfield = row['volumen_producto_planta_bakersfield'],
            dim_producto_ancho = row['dim_producto_ancho'], 
            dim_producto_largo = row['dim_producto_largo'],
            dim_producto_alto = row['dim_producto_alto']
        )
        for _, row in prod_op_merge.iterrows()
    ]
    
    # Crear un diccionario de cajas por producto desde el CSV de factibilidad
    cajas_por_producto = {}
    for _, row in factibilidad.iterrows():
        codigo = row['codigo_producto']
        cajas_str = row['cajas_asignables_id']
        
        if isinstance(cajas_str, str) and cajas_str:
            # Separar por '; ' y limpiar
            cajas_list = [c.strip() for c in cajas_str.split(';') if c.strip()]
            cajas_por_producto[codigo] = cajas_list

    # Recorrer la lista de productos en orden y agregar las cajas
    for producto in productos:
        if producto.codigo_producto in cajas_por_producto:
            for caja_id in cajas_por_producto[producto.codigo_producto]:
                producto.agregar_caja_asignable(caja_id)
                
    # Elegir grosor
    for caja in cajas:
        caja.elegir_grosor(grosor_mm=grosor)
    return cajas, productos

#### **Greedy 0: Costos base sin descuentos**

Inicialmente, plantearemos una solución únicamente comparando los costos unitarios base, sin considerar todavía los descuentos por volúmenes anuales.

Como no hay ninguna restricción sobre la cantidad de cajas que podemos comprar de cada tipo, el problema se reduce en encontrar para cada producto el tipo de caja que más le convenga, es decir, el que ofrezca el menor costo.

Empecemos eligiendo un grosor de 5mm para todos los tipos de cajas, de manera que la restricción de headspace no supone ningún problema, pues las dimensiones internas de las cajas se diferencian hasta un 10% de las originales.

In [45]:
cajas, productos = guardar_cajas_y_productos(grosor=4.5)

Armamos una función general para buscar un tipo de caja por su id.

In [46]:
def buscar_caja_por_id(id):
    caja_a_buscar = None
    for caja in cajas:
        if caja.caja_id == id:
            caja_a_buscar = caja
    return caja_a_buscar

In [47]:
solucion_base = Solucion(grosor=4.5)

for producto in productos:
    for caja_id in producto.cajas_asignables:
        caja = buscar_caja_por_id(caja_id)
        asignacion = Asignacion(producto,caja)
        solucion_base.agregar_asignacion(asignacion, descuentos=False)

#solucion_base.exportar_submmit(nombre_csv="1-base_5mm")
solucion_base.resumen_por_asignacion()

,codigo_producto,demanda_total,cant_cajas_asignables,caja_id,utilizacion_pallet,utilizacion_caja,descuento_buenos_aires,descuento_curitiba,descuento_santiago,descuento_monterrey,descuento_bakersfield,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,70,02cf77de65b70dd77905e2e33d78478f,0.385062,0.983137,0.0,0.0,0.0,0.0,0.0,1005298.45,85924,12888600,13893898.45
1,BR0001,1546613,70,0835ff365412a67b720a19713ec250f3,0.928963,1.089928,0.0,0.0,0.0,0.0,0.0,1005298.45,32223,4833450,5838748.45
2,BR0001,1546613,70,125888817c4da192ce9335454b8d4432,0.875156,1.158994,0.0,0.0,0.0,0.0,0.0,1005298.45,32223,4833450,5838748.45
3,BR0001,1546613,70,170f5916eaeb0e271ae012eedb1ad645,0.828083,1.016779,0.0,0.0,0.0,0.0,0.0,1005298.45,38667,5800050,6805348.45
4,BR0001,1546613,70,1fe6b9010b30d1d429fc0c7a240159aa,0.338855,1.121933,0.0,0.0,0.0,0.0,0.0,1005298.45,85924,12888600,13893898.45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22094,BR0421,145803,34,e70cd62d04659cbda3f1df24c7140e35,0.754876,1.222194,0.0,0.0,0.0,0.0,0.0,94771.95,2279,341850,436621.95
22095,BR0421,145803,34,f95687311339e4b9b0ef5622b26fd941,0.428443,1.071770,0.0,0.0,0.0,0.0,0.0,94771.95,4557,683550,778321.95
22096,BR0421,145803,34,fa8da38322149b23e14a120e71d38975,0.419599,0.953191,0.0,0.0,0.0,0.0,0.0,94771.95,5208,781200,875971.95
22097,BR0421,145803,34,fc6931875b8384c7bacd18bb4699795f,0.392084,1.022831,0.0,0.0,0.0,0.0,0.0,94771.95,5208,781200,875971.95


#### **Greedy 1: Elección por menor costo**

In [48]:
def solucion_greedy(grosor, cajas, productos_ordenados, criterio):

    solucion = Solucion(grosor=grosor)

    for producto in productos_ordenados:
        metricas = {}  # caja_id -> (volumen_total, costo_packaging, costo_flete, costo_total, utilizacion_pallet)
        
        for caja_id in producto.cajas_asignables:
            caja = buscar_caja_por_id(caja_id)
            
            # Volumen requerido actual (antes de asignar este producto)
            volumen_total = caja.unidades_total_requeridas()
            utilizacion_pallet = caja.utilizacion_pallet()
            
            # Costo simulado de asignar este producto a esta caja
            asignacion = Asignacion(producto, caja)
            caja.asignar_producto(producto)
            
            # Calculamos los costos del producto utilizando ese tipo de caja
            costo_packaging = asignacion.costo_packaging_producto_total()
            costo_flete = 150 * asignacion.cant_pallets_requeridas()
            costo_total = costo_packaging + costo_flete
            metricas[caja_id] = (volumen_total, costo_packaging, costo_flete, costo_total, utilizacion_pallet)
            
            # Revertimos la asignación para que no se aplique un descuento posterior
            caja.revocar_producto(producto)
        
        if criterio == "minimizar_costo_total":
            caja_id_optima = min(metricas, key=lambda id: (metricas[id][3]))
        elif criterio == "maximizar_volumen_tipo":        
            caja_id_optima = max(metricas, key=lambda id: (metricas[id][0], -metricas[id][3]))
        elif criterio == "minimizar_costo_flete":
            caja_id_optima = min(metricas, key=lambda id: (metricas[id][2]))
        elif criterio == "maximizar_utilizacion_pallet":
            caja_id_optima = max(metricas, key=lambda id: (metricas[id][4]))
        
        caja_optima = buscar_caja_por_id(caja_id_optima)
        
        asignacion = Asignacion(producto, caja_optima)
        solucion.agregar_asignacion(asignacion)    
    
    return solucion

Guardamos nuevamente cajas y productos, reiniciando toda asignación previa:

In [49]:
cajas, productos = guardar_cajas_y_productos(grosor=3)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: len(p.cajas_asignables))

solucion_greedy1 = solucion_greedy(3, cajas, productos_ordenados, criterio="minimizar_costo_total")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy1.resumen_por_asignacion()
solucion_greedy1.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 3mm
Número de tipos de cajas distintos: 24
Costo packaging: 26762351.160000004
Costo flete: 129933000
Costo total: 156695351.16
Utilización de pallet promedio: 0.887607186678943
Utilización de caja promedio: 1.274734617139488


In [50]:
solucion_greedy1.resumen_por_asignacion()

,codigo_producto,demanda_total,cant_cajas_asignables,caja_id,utilizacion_pallet,utilizacion_caja,descuento_buenos_aires,descuento_curitiba,descuento_santiago,descuento_monterrey,descuento_bakersfield,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,70,225c1760c0e9639bbb73f58d280b7a0e,0.892208,1.292433,0.1,-0.3,-0.3,-0.1,-0.2,660171.54,27619,4142850,4803021.54
1,BR0002,139211,73,1333ec917a8a6111a3fac88649307892,0.942483,1.272778,-0.3,-0.3,-0.3,0.1,-0.3,58468.62,2176,326400,384868.62
2,BR0003,172506,34,25b51ca97d24c6dd7fdc2fd6f9551dac,0.829900,1.328989,-0.2,-0.3,-0.3,-0.1,-0.3,72452.52,1797,269550,342002.52
3,BR0004,271715,75,161332db17a4c8bcba8a5651098cbe91,0.941168,1.240083,0.1,-0.3,-0.3,0.1,-0.3,114120.30,3774,566100,680220.30
4,BR0005,7586,95,161332db17a4c8bcba8a5651098cbe91,0.941168,1.322476,0.1,-0.3,-0.3,0.1,-0.3,3186.12,106,15900,19086.12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
422,BR0417,2761,42,1e4959b0a5703ba491bdc303578ad11c,0.838897,1.320153,-0.3,-0.3,-0.3,-0.3,-0.2,1159.62,35,5250,6409.62
423,BR0418,3942,42,1e4959b0a5703ba491bdc303578ad11c,0.838897,1.320153,-0.3,-0.3,-0.3,-0.3,-0.2,1655.64,50,7500,9155.64
424,BR0419,43300,46,3068e6f2a2ce79314e458b7a6ab57b4d,0.965300,1.257037,-0.3,-0.3,-0.2,-0.2,-0.3,20784.00,452,67800,88584.00
425,BR0420,205852,22,1e4959b0a5703ba491bdc303578ad11c,0.838897,1.214286,-0.3,-0.3,-0.3,-0.3,-0.2,86457.84,2574,386100,472557.84


In [51]:
cajas, productos = guardar_cajas_y_productos(grosor=3)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: len(p.cajas_asignables))

solucion_greedy2 = solucion_greedy(3, cajas, productos_ordenados, criterio="minimizar_costo_flete")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy2.resumen_por_asignacion()
solucion_greedy2.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 3mm
Número de tipos de cajas distintos: 29
Costo packaging: 26859305.4
Costo flete: 129912750
Costo total: 156772055.4
Utilización de pallet promedio: 0.8877285133296356
Utilización de caja promedio: 1.2748343130077449


In [52]:
cajas, productos = guardar_cajas_y_productos(grosor=3)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: len(p.cajas_asignables))

solucion_greedy3 = solucion_greedy(3, cajas, productos_ordenados, criterio="maximizar_utilizacion_pallet")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy3.resumen_por_asignacion()
solucion_greedy3.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 3mm
Número de tipos de cajas distintos: 14
Costo packaging: 26430939.06000001
Costo flete: 137337750
Costo total: 163768689.06
Utilización de pallet promedio: 0.9174022395832455
Utilización de caja promedio: 1.162152773742875


In [53]:
solucion_greedy3.resumen_por_asignacion()


,codigo_producto,demanda_total,cant_cajas_asignables,caja_id,utilizacion_pallet,utilizacion_caja,descuento_buenos_aires,descuento_curitiba,descuento_santiago,descuento_monterrey,descuento_bakersfield,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,70,b4d42409ba0a801454f8f3bc60d94ec1,0.962082,1.021655,0.1,-0.3,-0.3,-0.1,-0.2,660171.54,32223,4833450,5493621.54
1,BR0002,139211,73,79637c70988d89fe3abe191b59c1ed0c,0.965300,1.242207,-0.3,-0.3,-0.3,0.1,-0.3,58468.62,2176,326400,384868.62
2,BR0003,172506,34,dc0b41c3254944ee2f8977c80dc8f455,0.974929,0.933211,-0.3,-0.3,-0.3,-0.3,-0.3,72452.52,2157,323550,396002.52
3,BR0004,271715,75,614801e8bbdec32546c0d92055dfb596,0.965300,1.208122,-0.3,-0.3,-0.3,-0.2,-0.3,114120.30,3774,566100,680220.30
4,BR0005,7586,95,614801e8bbdec32546c0d92055dfb596,0.965300,1.288392,-0.3,-0.3,-0.3,-0.2,-0.3,3186.12,106,15900,19086.12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
422,BR0417,2761,42,83d3c7c10fbc6fe2e476fd8c600855e2,0.867825,1.143225,-0.3,-0.3,-0.3,-0.3,-0.3,1159.62,39,5850,7009.62
423,BR0418,3942,42,83d3c7c10fbc6fe2e476fd8c600855e2,0.867825,1.143225,-0.3,-0.3,-0.3,-0.3,-0.3,1655.64,55,8250,9905.64
424,BR0419,43300,46,dc0b41c3254944ee2f8977c80dc8f455,0.974929,1.030023,-0.3,-0.3,-0.3,-0.3,-0.3,18186.00,542,81300,99486.00
425,BR0420,205852,22,83d3c7c10fbc6fe2e476fd8c600855e2,0.867825,1.051546,-0.3,-0.3,-0.3,-0.3,-0.3,86457.84,2860,429000,515457.84


In [54]:
cajas, productos = guardar_cajas_y_productos(grosor=3)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: len(p.cajas_asignables))

solucion_greedy4 = solucion_greedy(3, cajas, productos_ordenados, criterio="maximizar_volumen_tipo")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy4.resumen_por_asignacion()
solucion_greedy4.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 3mm
Número de tipos de cajas distintos: 12
Costo packaging: 26500385.40000001
Costo flete: 150459900
Costo total: 176960285.4
Utilización de pallet promedio: 0.8276317266624049
Utilización de caja promedio: 1.1980568453546432


In [55]:
cajas, productos = guardar_cajas_y_productos(grosor=3)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: p.demanda_total(), reverse=True)

solucion_greedy5 = solucion_greedy(3, cajas, productos_ordenados, criterio="minimizar_costo_total")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy5.resumen_por_asignacion()
solucion_greedy5.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 3mm
Número de tipos de cajas distintos: 25
Costo packaging: 26672737.98000001
Costo flete: 129912750
Costo total: 156585487.98000002
Utilización de pallet promedio: 0.8917564679316201
Utilización de caja promedio: 1.2690092487354785


In [56]:
cajas, productos = guardar_cajas_y_productos(grosor=3)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: p.demanda_total(), reverse=True)

solucion_greedy6 = solucion_greedy(3, cajas, productos_ordenados, criterio="minimizar_costo_flete")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy6.resumen_por_asignacion()
solucion_greedy6.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 3mm
Número de tipos de cajas distintos: 29
Costo packaging: 26859305.400000017
Costo flete: 129912750
Costo total: 156772055.4
Utilización de pallet promedio: 0.8877285133296358
Utilización de caja promedio: 1.2748343130077429


In [57]:
cajas, productos = guardar_cajas_y_productos(grosor=3)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: p.demanda_total(), reverse=True)

solucion_greedy7 = solucion_greedy(3, cajas, productos_ordenados, criterio="maximizar_utilizacion_pallet")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy7.resumen_por_asignacion()
solucion_greedy7.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 3mm
Número de tipos de cajas distintos: 14
Costo packaging: 26430939.060000025
Costo flete: 137337750
Costo total: 163768689.06000003
Utilización de pallet promedio: 0.9174022395832456
Utilización de caja promedio: 1.1621527737428754


In [58]:
cajas, productos = guardar_cajas_y_productos(grosor=3)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: p.demanda_total(), reverse=True)

solucion_greedy8 = solucion_greedy(3, cajas, productos_ordenados, criterio="maximizar_volumen_tipo")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy8.resumen_por_asignacion()
solucion_greedy8.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 3mm
Número de tipos de cajas distintos: 13
Costo packaging: 26427599.10000004
Costo flete: 150588600
Costo total: 177016199.10000002
Utilización de pallet promedio: 0.8385121543197981
Utilización de caja promedio: 1.1807830415639484
